# Step 8 — Mask-based Baselines (EoMT)
Evaluates MSP, MaxLogit, MaxEntropy, RbA on 3 checkpoints (Cityscapes, COCO, Finetuned).
Also evaluates Temperature Scaling on MSP.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

REPO_URL = "https://github.com/flaviofrasca/MaskArchitectureAnomaly_CourseProject.git"
REPO_DIR = "/content/MaskArchitectureAnomaly_CourseProject"
BRANCH   = "step_8"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}
    print("Repo aggiornato")

# eval_anomaly_eomt.py deve girare da eomt/ perche importa i moduli locali
%cd {REPO_DIR}/eomt

Cloning into '/content/MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 285, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (82/82), done.
remote: Total 285 (delta 61), reused 52 (delta 48), pack-reused 155 (from 2)
Receiving objects: 100% (285/285), 27.72 MiB | 13.82 MiB/s, done.
Resolving deltas: 100% (86/86), done.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [3]:
!pip install ood-metrics timm lightning transformers torchmetrics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 101.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 60.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_versio

## Checkpoint Cityscapes

In [ ]:
!python eval_anomaly_eomt.py --model_type cityscapes --method MSP
!python eval_anomaly_eomt.py --model_type cityscapes --method MaxLogit
!python eval_anomaly_eomt.py --model_type cityscapes --method MaxEntropy
!python eval_anomaly_eomt.py --model_type cityscapes --method RbA

## Checkpoint COCO

In [4]:
!python eval_anomaly_eomt.py --model_type coco --method MSP
!python eval_anomaly_eomt.py --model_type coco --method MaxLogit
!python eval_anomaly_eomt.py --model_type coco --method MaxEntropy
!python eval_anomaly_eomt.py --model_type coco --method RbA


Loading EoMT [coco] ...
model.safetensors: 100% 346M/346M [00:06<00:00, 56.3MB/s]
Loaded weights: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/coco/eomt_coco.bin

Model: EoMT [coco]  |  Method: MSP
Dataset                 AuPRC    FPR95
----------------------------------------
  Evaluating: SMIYC RA-21 ...
  SMIYC RA-21          AuPRC:   38.88%   FPR95:   77.50%
  Evaluating: SMIYC RO-21 ...
  SMIYC RO-21          AuPRC:    2.76%   FPR95:   99.98%
  Evaluating: FS L&F ...
  FS L&F               AuPRC:    3.78%   FPR95:   97.21%
  Evaluating: FS Static ...
  FS Static            AuPRC:    4.67%   FPR95:   99.48%
  Evaluating: Road Anomaly ...
  Road Anomaly         AuPRC:   18.86%   FPR95:   91.84%

Results appended to results_eomt.txt

Loading EoMT [coco] ...
Loaded weights: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/coco/eomt_coco.bin

Model: EoMT [coco]  |  Method: MaxLo

## Checkpoint Finetuned

In [ ]:
!python eval_anomaly_eomt.py --model_type finetuned --method MSP
!python eval_anomaly_eomt.py --model_type finetuned --method MaxLogit
!python eval_anomaly_eomt.py --model_type finetuned --method MaxEntropy
!python eval_anomaly_eomt.py --model_type finetuned --method RbA

## Temperature Scaling (MSP)

PRO TIP: il modello gira **una volta sola** per checkpoint e salva i logits su disco.
Le 4 temperature vengono valutate caricando i logits cached, senza rieseguire il forward pass.

**mIoU**: non disponibile sui dataset anomaly (nessuna GT semantica) — colonna N/A.

In [ ]:
# Cityscapes — run once: caches logits + evaluates all temperatures
!python eval_temperature_scaling.py --model_type cityscapes

In [17]:
# PATCH per il salvataggio delle immagini in cache (da 2048x1024 a 640x640)
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

OLD = """        H, W = pixel_logits.shape[1], pixel_logits.shape[2]
        if ood_gts.shape != (H, W):
            ood_gts = np.array(
                Image.fromarray(ood_gts).resize((W, H), Image.NEAREST)
            )

        np.save(logits_path, pixel_logits.cpu().numpy().astype(np.float16))
        np.save(gt_path, ood_gts)"""

NEW = """        # Resize to _RESIZE_TO at save time (~5x smaller cache)
        TH, TW = _RESIZE_TO
        pixel_logits = F.interpolate(
            pixel_logits.unsqueeze(0), size=(TH, TW), mode='bilinear', align_corners=False
        ).squeeze(0)
        ood_gts = np.array(Image.fromarray(ood_gts).resize((TW, TH), Image.NEAREST))

        np.save(logits_path, pixel_logits.cpu().numpy().astype(np.float16))
        np.save(gt_path, ood_gts)"""

if OLD in code:
    code = code.replace(OLD, NEW)
    with open(filepath, 'w') as f:
        f.write(code)
    print("Patch OK")
else:
    print("Pattern non trovato — verifica indentazione")


Pattern non trovato — verifica indentazione


In [28]:
# AUTOCLICKER
%%javascript
function keepAlive() {
    console.log("Keep-alive: " + new Date().toLocaleTimeString());
    window.scrollBy(0, 1);
    window.scrollBy(0, -1);
    setTimeout(keepAlive, 60000);
}
keepAlive();

<IPython.core.display.Javascript object>

In [23]:
# PATCH completa (???)
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

# Patch 1: aggiungi --cache_dir all'argparse
code = code.replace(
    "help='Re-run inference even if cache exists')",
    "help='Re-run inference even if cache exists')\n    parser.add_argument('--cache_dir', default=None,\n                        help='Directory per i logit cachati')"
)

# Patch 2: usa --cache_dir se fornito
code = code.replace(
    'cache_dir = f"logits_cache_{args.model_type}"',
    'cache_dir = args.cache_dir if args.cache_dir else f"logits_cache_{args.model_type}"'
)

with open(filepath, 'w') as f:
    f.write(code)

# Verifica
import subprocess
result = subprocess.run(['python', filepath, '--help'], capture_output=True, text=True)
print(result.stdout[-500:])


In [22]:
# COCO (con cache su drive)
CACHE_DIR = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco"
!python eval_temperature_scaling.py --model_type coco --cache_dir {CACHE_DIR}




Loading EoMT [coco] for logits caching ...
Loaded weights: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/coco/eomt_coco.bin
  Caching: SMIYC RA-21 ...
    10 images saved.
  Caching: SMIYC RO-21 ...
    30 images saved.
  Caching: FS L&F ...
    99 images saved.
  Caching: FS Static ...
    20 images saved.
  Caching: Road Anomaly ...
    60 images saved.

Logits cached in: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco/

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/npyio.py", line 456, in load
    return format.read_array(fid, allow_pickle=allow_pickle,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/lib/format.py", line 839, in read_a

In [24]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import glob, os, numpy as np

cache_dir = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco"
EXPECTED_SIZE = 133 * 640 * 640 * 2  # float16, bytes

logit_files = sorted(glob.glob(f"{cache_dir}/*_logits.npy"))
print(f"File totali: {len(logit_files)}")

corrupted = []
for f in logit_files:
    size = os.path.getsize(f)
    if size < EXPECTED_SIZE * 0.9:  # tolleranza 10%
        corrupted.append((f, size / 1e6))

print(f"File corrotti: {len(corrupted)}")
for f, mb in corrupted:
    print(f"  {mb:.1f} MB  {os.path.basename(f)}")


Mounted at /content/drive
File totali: 219
File corrotti: 0


In [25]:
CACHE_DIR = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco"
!python eval_temperature_scaling.py --model_type coco --cache_dir {CACHE_DIR}


Traceback (most recent call last):
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 297, in <module>
    main()
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 198, in main
    parser.add_argument('--cache_dir', default=None,
  File "/usr/lib/python3.12/argparse.py", line 1500, in add_argument
    return self._add_action(action)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1885, in _add_action
    self._optionals._add_action(action)
  File "/usr/lib/python3.12/argparse.py", line 1705, in _add_action
    action = super(_ArgumentGroup, self)._add_action(action)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/argparse.py", line 1514, in _add_action
    self._check_conflict(action)
  File "/usr/lib/python3.12/argparse.py", line 1654, in _check_conflict
    conflict_handler(action, confl_optionals)
  File "/usr/lib/python

In [26]:
filepath = "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py"
with open(filepath) as f:
    code = f.read()

DUPLICATE = "\n    parser.add_argument('--cache_dir', default=None,\n                        help='Directory per i logit cachati')"
count = code.count(DUPLICATE)
print(f"Occorrenze trovate: {count}")

# Rimuove solo la seconda occorrenza
code = code.replace(DUPLICATE, "", 1)  # rimuove la prima, ne resta una
with open(filepath, 'w') as f:
    f.write(code)
print("Fix OK")


Occorrenze trovate: 2
Fix OK


In [27]:
CACHE_DIR = "/content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco"
!python eval_temperature_scaling.py --model_type coco --cache_dir {CACHE_DIR}



Using cached logits: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/logits_cache_coco/  (219 files)

Model: EoMT [coco]  |  Temperature Scaling on MSP
Loading logits into memory and evaluating all temperatures ...
  RA-21 ...
  RO-21 ...
  L&F ...
^C


In [5]:
# COCO
!python eval_temperature_scaling.py --model_type coco


Loading EoMT [coco] for logits caching ...
Loaded weights: /content/drive/.shortcut-targets-by-id/1osgiWms0a4SYz1evCZNwMV-jv--0I0RU/MaskArch_Shared/checkpoints/coco/eomt_coco.bin
  Caching: SMIYC RA-21 ...
    10 images saved.
  Caching: SMIYC RO-21 ...
    30 images saved.
  Caching: FS L&F ...
Traceback (most recent call last):
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 292, in <module>
    main()
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 208, in main
    n = cache_logits(model, name, full_pattern, cache_dir, device)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/MaskArchitectureAnomaly_CourseProject/eomt/eval_temperature_scaling.py", line 88, in cache_logits
    ood_gts = 

In [ ]:
# Finetuned
!python eval_temperature_scaling.py --model_type finetuned

## Risultati

In [ ]:
REPO_DIR = "/content/MaskArchitectureAnomaly_CourseProject"
print("=== MSP / MaxLogit / MaxEntropy / RbA ===")
with open(f"{REPO_DIR}/eomt/results_eomt.txt") as f:
    print(f.read())

print("\n=== Temperature Scaling ===")
with open(f"{REPO_DIR}/eomt/results_temperature_scaling.txt") as f:
    print(f.read())